In [ ]:
from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex

reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(parsed_docs)

Highlight feature para recoger cachos que me indiquen en que partes hay que buscar el documento.
Key parameters: - highlight_fields: Which fields to extract snippets from - max_matches: How many snippets to show per document (3 = not too many) - snippet_size: How long each snippet should be (50 tokens = enough to understand relevance) - stop_words: Words to ignore (like "evidently" which appears everywhere) - stemmer: Matches word variations (use → using, etc.)

In [ ]:
from minsearch import Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS

In [ ]:
stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"evidently"}

highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer="snowball", stop_words=stopwords)
)

In [ ]:
query = "how can I create evidently dahsbord?"

In [ ]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

In [ ]:
search_results = search(query)

In [ ]:
highlighter.highlight(query, search_results)

[{'title': 'Evidently Cloud',
  'description': 'How to set up Evidently Cloud account.',
  'icon': 'cloud',
  'content': {'matches': ['... a SaaS product, but you **can** self-host the open-sourc...',
    '...er tools.\n</Note>\n\n## 1. **Create** an Account\n\n- If not yet...',
    '...up).\n- After logging in, **create** an **Organization** and ...'],
   'total_matches': 6},
  'filename': 'docs/setup/cloud.mdx'},
 {'title': 'LLM as a judge',
  'description': 'How to create and evaluate an LLM judge.',
  'content': {'matches': ['...n demonstrating **how to **create** and tune the LLM evaluat...',
    '...M evaluator**, which you **can** then apply in different ...',
    "...re's what we'll do:\n\n- ****Create** an evaluation dataset**...."],
   'total_matches': 64},
  'filename': 'examples/LLM_judge.mdx'},
 {'title': 'What is Evidently?',
  'description': 'Welcome to the Evidently documentation.',
  'mode': 'wide',
  'content': {'matches': ['...data and AI systems. You **can** use it 

Necesitamos una forma de retornar un documento por nombre del indice

In [ ]:
file_index = {doc["filename"]: doc["content"] for doc in parsed_docs}

In [ ]:
file_index["docs/setup/self-hosting.mdx"]

'<Note>\n  Evidently Cloud is no longer available as a SaaS product, but you can self-host the open-source Evidently Platform (as described on this page), or use the Evidently library with other tools.\n</Note>\n\nIn addition to using Evidently Python library, you can self-host the UI Service to get a monitoring Dashboard and organize the results of your evaluations. This is optional: you can also view evaluation results in Python or export to JSON or HTML.\n\nTo get a self-hostable Dashboard, you must:\n\n- Create a Workspace (local or remote) to store your data.\n- Launch the UI service.\n\n## 1. Create a Workspace\n\nOnce you [install Evidently](/docs/setup/installation), you will need to create a `workspace`. This designates a remote or local directory where you will store the evaluation results (JSON Reports called `snapshots`), traces or datasets. The UI Service will read the data from this source.\n\nAs a storage backend, Evidently supports:\n\n- a file system\n- any SQL-like da

Creamos las herramientas de busqueda

In [ ]:
from typing import Any, Dict, List

class SearchTools:
    """
    Provides search and file retrieval utilities over an indexed data store.
    """
    def __init__(self, index: Any, highlighter: Any, file_index: Dict[str, str]) -> None:
        """
        Initialize the SearchTools instance.
        Args:
            index: Searchable index providing a `search` method.
            highlighter: Highlighter used to annotate search results.
            file_index (Dict[str, str]): Mapping of filenames to file contents.
        """
        self.index = index
        self.highlighter = highlighter
        self.file_index = file_index

    def search(self, query: str) -> List[Dict[str, Any]]:
        """
        Search the index for results matching a query and highlight them.
        Args:
            query (str): The search query to look up in the index.
        Returns:
            List[Dict[str, Any]]: A list of highlighted search result objects.
        """
        search_results = self.index.search(query=query, num_results=5)
        return self.highlighter.highlight(query, search_results)

    def get_file(self, filename: str) -> str:
        """
        Retrieve a file's contents by filename.
        Args:
            filename (str): The filename of the file to retrieve.
        Returns:
            str: The file contents if found, otherwise an error message.
        """
        if filename in self.file_index:
            return self.file_index[filename]
        return f"file {filename} does not exist"

Y creamos la instancia de la clase

In [ ]:
search_tools = SearchTools(index, highlighter, file_index)

Creamos las instrucciones del agente

In [ ]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2â€“3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2â€“3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

In [ ]:
from toyaikit.tools import Tools

agent_tools = Tools()
agent_tools.add_tools(search_tools)

agent_tools.get_tools()

[{'type': 'function',
  'name': 'get_file',
  'description': "Retrieve a file's contents by filename.\nArgs:\n    filename (str): The filename of the file to retrieve.\nReturns:\n    str: The file contents if found, otherwise an error message.",
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'filename parameter'}},
   'required': ['filename'],
   'additionalProperties': False}},
 {'type': 'function',
  'name': 'search',
  'description': 'Search the index for results matching a query and highlight them.\nArgs:\n    query (str): The search query to look up in the index.\nReturns:\n    List[Dict[str, Any]]: A list of highlighted search result objects.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [ ]:

from openai import OpenAI
from toyaikit.llm import OpenAIClient
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface)

Creamos el agente

In [ ]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

Y lo ejecutamos

In [ ]:
result = agent.loop(
    query + " show me the code",
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


In [ ]:
tools = [search_tools.search, search_tools.get_file]
tools

In [ ]:
from toyaikit.tools import get_instance_methods

tools = get_instance_methods(search_tools)

In [ ]:
from pydantic_ai import Agent
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [ ]:
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

In [ ]:
query = 'how do I use evidently to monitor my machine learning models?'
result = await search_agent.run(query)
print(result.output)

In [ ]:
messages = result.all_messages()
messages

In [ ]:
result.usage

In [ ]:
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

In [ ]:
new_prompt = 'show me the code'
result2 = await search_agent.run(new_prompt, message_history=messages)
print(result2.output)

In [ ]:
def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == 'user-prompt':
                print('USER:', p.content)
            if part_kind == 'tool-call':
                print('TOOL CALL:', p.tool_name, p.args)
            if part_kind == 'tool-return':
                print('TOOL RETURN:', p.tool_name)
            if part_kind == 'text':
                print(p.content)
        print()

In [ ]:
print_messages(result2.new_messages())

In [ ]:
from pydantic_ai import RunUsage
messages = []
usage = RunUsage()

while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    result = await search_agent.run(user_prompt, message_history=messages)
    usage = usage + result.usage

    print_messages(result.new_messages())
    messages.extend(result.new_messages())
usage

In [ ]:
from pydantic_ai.messages import FunctionToolCallEvent

class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        # Detect nested streams
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)

In [ ]:
callback = NamedCallback(search_agent)

In [ ]:
result = await search_agent.run(
    "how do I build dashboads with evidently?",
    event_stream_handler=callback
)
result